# PLS — interpretação

O VIP vem de um único `PLSRegression(scale=True)` no treino interno, com o número de componentes escolhido na validação. Ele não é a média dos ajustes diários. Preditores de PM2.5 muito correlacionados dividem o crédito: um VIP menor não significa que a coluna seja inútil sozinha.

VIP acima de 1 indica preditor mais útil que a média, nesse ajuste, para acompanhar o PM2.5 da hora seguinte. As estações externas entram com a medição da origem, nunca com o valor da hora prevista.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

current = Path.cwd()
while current != current.parent and not (current / "bases" / "grupo3").exists():
    current = current.parent
ROOT = current
BASE = ROOT / "analyses" / "grupo3"
OUT = BASE / "03_resultados" / "outputs"
print(ROOT)

vip = pd.read_csv(OUT / "pls_vip.csv")
notes = pd.read_csv(OUT / "residuos_interpretacao.csv")
ljung = pd.read_csv(OUT / "ljung_box_results.csv")
print(vip.to_string(index=False))
print()
print(notes.loc[notes["model"] == "PLS_Regression", "interpretacao"].iloc[0])
print(ljung.loc[ljung["model"] == "PLS_Regression"].to_string(index=False))


/Users/leticia.nsilva/Desktop/personal/pls-regration-comparation
          feature      vip
           pm25_t 1.520194
 pm25_roll_mean_3 1.430092
         pm25_t_1 1.410636
     Wanliu_PM2.5 1.410418
         pm25_t_2 1.355311
  Changping_PM2.5 1.290258
       local_PM10 1.279419
         local_CO 1.110642
pm25_roll_mean_24 1.108326
        pm25_t_23 0.687709
       local_WSPM 0.445223
       local_DEWP 0.382990
       local_TEMP 0.329839
  target_hour_sin 0.223951
   target_dow_sin 0.179299
  target_hour_cos 0.175694
   target_dow_cos 0.156230

O erro médio é 0.55 µg/m³, então a previsão fica abaixo do observado. O desvio-padrão do resíduo é 17.60 µg/m³. No lag 24, ainda há autocorrelação nos resíduos (p=4.09e-15).
         model  lag    lb_stat    lb_pvalue  n_residuals
PLS_Regression   10  76.341629 2.606269e-12         6023
PLS_Regression   24 122.101757 4.091276e-15         6023
PLS_Regression   48 172.980250 5.098465e-16         6023


## Como ler o erro

O resíduo é observado menos previsão. Média positiva significa que o modelo ficou abaixo do PM2.5 realizado. O desvio-padrão do resíduo mede a dispersão. O Ljung-Box no lag 24 pergunta se, um dia depois, o erro ainda carrega o erro anterior. Rejeitar a hipótese nula não muda o MAE; mostra que o erro ainda tem memória.